# Modelagem Supervisionada

In [12]:
#importando bibliotecas
import pandas as pd
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
import plotly.figure_factory as ff 
from plotly.subplots import make_subplots

import pandas as pd
from sklearn.cluster import KMeans
from xgboost import XGBClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, roc_auc_score

pd.set_option('display.max_columns', None)

In [13]:
#importando dados
df = pd.read_csv('../data/curated/features.csv')
df.head()

,MONTH,DAY_OF_WEEK,SCHEDULED_DEPARTURE_HOUR,PERIOD,SEASON,DISTANCE,AIRPORT_PROFILE,AIRLINE_PROFILE,IS_DELAYED
0,0.253167,-1.456333,1.135593,1.397838,0.625861,1.580078,1.428664,1.432931,1
1,1.436686,-0.953642,0.716535,0.161628,1.588467,-1.216751,-1.374316,-0.434042,1
2,-1.522113,0.554430,1.135593,1.397838,-1.299350,-1.014511,0.494337,-1.367529,1
3,-1.226233,1.559811,-0.121581,0.161628,-1.299350,0.003265,0.494337,-0.434042,0
4,1.436686,1.559811,-0.331110,0.161628,1.588467,0.049304,-1.374316,0.499444,0


Treinamento

In [14]:
X = df.drop(columns=['IS_DELAYED'])
y = df['IS_DELAYED']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f'Distribuição das classes:\n{y.value_counts(normalize=True)}')
print(f'Treino: {X_train.shape}, Teste: {X_test.shape}')

Distribuição das classes:
IS_DELAYED
1    0.5
0    0.5
Name: proportion, dtype: float64
Treino: (1543796, 8), Teste: (385950, 8)


In [15]:
print('Treinando Regressão Logística...')
lr_model = LogisticRegression(max_iter=1000)
lr_model.fit(X_train, y_train)

y_pred_lr = lr_model.predict(X_test)
print('\nResultado: Regressão Logística')
print(classification_report(y_test, y_pred_lr))

Treinando Regressão Logística...

Resultado: Regressão Logística
              precision    recall  f1-score   support

           0       0.59      0.58      0.58    192975
           1       0.58      0.59      0.59    192975

    accuracy                           0.58    385950
   macro avg       0.58      0.58      0.58    385950
weighted avg       0.58      0.58      0.58    385950



In [16]:
print('Treinando XGBoost...')
xgb_model = XGBClassifier(
    n_estimators=500,
    max_depth=10,
    learning_rate=0.01,
    subsample=0.8,
    colsample_bytree=0.8,
    tree_method='hist',
    random_state=42,
    n_jobs=-1   
)
xgb_model.fit(X_train, y_train)

y_pred_xgb = xgb_model.predict(X_test)
print('\nResultado:')
print(classification_report(y_test, y_pred_xgb))
print(f'ROC-AUC Score: {roc_auc_score(y_test, xgb_model.predict_proba(X_test)[:, 1]):.4f}')

Treinando XGBoost...

Resultado:
              precision    recall  f1-score   support

           0       0.63      0.59      0.61    192975
           1       0.61      0.65      0.63    192975

    accuracy                           0.62    385950
   macro avg       0.62      0.62      0.62    385950
weighted avg       0.62      0.62      0.62    385950

ROC-AUC Score: 0.6684


In [17]:
def plot_confusion_matrix(y_true, y_pred, title):
    cm = confusion_matrix(y_true, y_pred)
    x = ['Previsto Pontual', 'Previsto Atrasado']
    y = ['Real Pontual', 'Real Atrasado']
    
    fig = ff.create_annotated_heatmap(
        z=cm, 
        x=x, 
        y=y, 
        annotation_text=cm, 
        colorscale='Blues'
    )
    fig.update_layout(
        title=title,
        xaxis_title='Predição',
        yaxis_title='Realidade',
        template='plotly_white'
    )
    return fig

In [18]:
fig_xgb = plot_confusion_matrix(y_test, y_pred_xgb, 'Matriz de Confusão: XGBoost')
fig_xgb.show()

In [19]:
def plot_roc_curve(y_true, y_probs, title):
    fpr, tpr, _ = roc_curve(y_true, y_probs)
    auc_score = roc_auc_score(y_true, y_probs)

    fig = go.Figure()

    fig.add_trace(go.Scatter(
            x=fpr, 
            y=tpr,
            mode='lines',
            name=f'XGBoost (AUC = {auc_score:.4f})',
            line=dict(color='darkblue', width=3)
        )
    )

    fig.add_trace(go.Scatter(
            x=[0, 1], 
            y=[0, 1],
            mode='lines',
            name='Predição Aleatória',
            line=dict(color='red', dash='dash')
        )
    )

    fig.update_layout(
        title=title,
        xaxis_title='Taxa de Falso Positivo (1 - Especificidade)',
        yaxis_title='Taxa de Verdadeiro Positivo (Sensibilidade)',
        height=600,
        template='plotly_white'
    )
    
    fig.show()

    return auc_score

In [20]:
y_probs_xgb = xgb_model.predict_proba(X_test)[:, 1]
plot_roc_curve(y_test, y_probs_xgb, 'Curva ROC')

0.668376488130558

Que características aumentam a chance de atraso em um voo?

In [ ]:
df_importances = pd.DataFrame({
    'Variável': features,
    'Importância': xgb_model.feature_importances_
}).sort_values(by='Importância', ascending=False)

fig = px.bar(
    df_importances,
    x='Importância',
    y='Variável',
    orientation='h',
    title='Importância das Variáveis Explanatórias'
)
fig.show()

ValueError: Data must be 1-dimensional, got ndarray of shape (1929746, 8) instead